<a href="https://colab.research.google.com/github/hari-hara-sudharsan/AI-Bot/blob/main/Data_Cleasing_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning & Visualization Project: Teen Mental Health

## Download and Load Dataset

In [1]:
import kagglehub
import pandas as pd
import os

# Download latest version of the dataset
path = kagglehub.dataset_download("argonnxx/teen-mental-health")

print("Path to dataset files:", path)

100%|██████████| 19.0k/19.0k [00:00<00:00, 12.5MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/argonnxx/teen-mental-health/versions/1


In [2]:
# Assuming the CSV file is directly in the downloaded path or a subdirectory
# We need to find the actual CSV file name.
# Let's list the files in the downloaded directory to find the CSV.

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if csv_files:
    # Assuming there's only one relevant CSV file or we pick the first one
    data_file_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(data_file_path)
    print(f"Loaded dataset from: {data_file_path}")
    display(df.head())
else:
    print("No CSV files found in the dataset directory.")

Loaded dataset from: /root/.cache/kagglehub/datasets/argonnxx/teen-mental-health/versions/1/Teen_Mental_Health.csv


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,mental_health_risk_score,sleep_quality,digital_wellbeing_flag
0,14,male,7.9,Facebook,7.4,2.9,3.01,1.5,low,2,2,1,0,5,Fair,At Risk
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0,19,Good,Moderate
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0,8,Fair,Healthy
3,15,male,7.4,YouTube,6.9,1.6,3.48,0.8,medium,1,7,9,0,17,Fair,Moderate
4,15,female,4.7,All Platforms,4.9,3.0,2.37,1.4,medium,3,5,2,0,10,Poor,Moderate


## Initial Data Exploration

In [3]:
print("DataFrame Info:")
df.info()

print("\nDataFrame Description (Numerical Columns):")
display(df.describe())

print("\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False))

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   object 
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   object 
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   object 
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
 13  mental_health_risk_score  1200 non-null   int64

,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,stress_level,anxiety_level,addiction_level,depression_label,mental_health_risk_score
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,15.928333,4.536667,6.449417,1.740333,2.990383,1.014500,5.445833,5.636667,5.565000,0.025833,16.647500
std,2.021947,2.029599,1.442677,0.716660,0.576758,0.582185,2.903290,2.859453,2.830627,0.158704,5.038128
min,13.000000,1.000000,4.000000,0.500000,2.000000,0.000000,1.000000,1.000000,1.000000,0.000000,3.000000
25%,14.000000,2.800000,5.200000,1.100000,2.500000,0.500000,3.000000,3.000000,3.000000,0.000000,13.000000
50%,16.000000,4.500000,6.500000,1.800000,2.990000,1.000000,5.000000,6.000000,6.000000,0.000000,17.000000
75%,18.000000,6.300000,7.600000,2.400000,3.480000,1.500000,8.000000,8.000000,8.000000,0.000000,20.000000
max,19.000000,8.000000,9.000000,3.000000,4.000000,2.000000,10.000000,10.000000,10.000000,1.000000,30.000000



Missing values per column:


,0
age,0
gender,0
daily_social_media_hours,0
platform_usage,0
sleep_hours,0
screen_time_before_sleep,0
academic_performance,0
physical_activity,0
social_interaction_level,0
stress_level,0


## Check for Duplicates

In [4]:
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

if num_duplicates > 0:
    print("Displaying first 5 duplicate rows (if any):")
    display(df[df.duplicated(keep=False)].head())
    # Option to drop duplicates if desired:
    # df.drop_duplicates(inplace=True)
    # print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicate rows found.")

Number of duplicate rows: 0
No duplicate rows found.


## Outlier Detection and Handling

In [5]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

print("Detecting outliers using IQR method:\n")

outlier_summary = {} # To store counts of outliers per column

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    num_outliers = len(outliers)

    if num_outliers > 0:
        outlier_summary[col] = num_outliers
        print(f"Column '{col}': {num_outliers} outliers detected.")
        # Optional: display some outlier rows for inspection
        # display(outliers[[col]].head())

if not outlier_summary:
    print("No significant outliers detected using the IQR method in numerical columns.")
else:
    print("\nSummary of outliers detected:")
    for col, count in outlier_summary.items():
        print(f"- {col}: {count} outliers")

# Note: Handling outliers (e.g., capping, removing) depends on the specific analysis goals.
# For this project, we'll focus on identification for now.

Detecting outliers using IQR method:

Column 'depression_label': 31 outliers detected.

Summary of outliers detected:
- depression_label: 31 outliers
